# Demo: varying clonal shapes

Generate six Visium-like spatial architectures on a fixed hexagonal lattice
(**1000** normal / **300** tumor1 / **700** tumor2 spots).

## Requirements
`numpy`, `pandas`, `matplotlib`

## Paths (repository root)
| Role | Path |
|------|------|
| **Input** (optional) | `data/cell_anno.tsv` |
| **Output** | `output/vary_shape/<condition>/` |

### Input file
`data/cell_anno.tsv` — header-free TSV:
```
barcode<TAB>clone_label
```
`clone_label` ∈ `{normal, tumor1, tumor2}`. Need at least 1000 / 300 / 700 rows of each.
If the file is absent, synthetic barcodes (`normal_0000`, …) are used.

### Output files (per condition)
| File | Format |
|------|--------|
| `tissue_positions_list.csv` | no header; `barcode, in_tissue, x, y, pixel_row, pixel_col` |
| `spot_anno_pattern.tsv` | TSV with header; barcode index + `spot_anno` |
| `barcodes.tsv.gz` | one barcode per line |

Conditions: `Separated_clusters`, `Mixed`, `Ring`, `Stripes`, `Intermixing`, `Single_tumor_region`.

## Run
```bash
# from repository root
jupyter nbconvert --to notebook --execute 01_vary_shape.ipynb
```


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

# Run this notebook from the repository root (directory that contains this file).
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "_spatial_pattern_utils.py").exists():
    raise FileNotFoundError(
        "Cannot find _spatial_pattern_utils.py in cwd. "
        "cd to the demo repository root before running."
    )
sys.path.insert(0, str(REPO_ROOT))

from _spatial_pattern_utils import (
    DEFAULT_CELL_ANNO,
    DEFAULT_OUTPUT_DIR,
    SEED,
    TARGET_PER_TYPE,
    load_or_make_barcodes_by_type,
    make_hex_grid,
    map_barcodes_to_labels,
    plot_spatial,
    resolve_repo_root,
    write_pattern_dir,
)

REPO_ROOT = resolve_repo_root()
# Optional input: header-free TSV with columns barcode, clone_label
CELL_ANNO = REPO_ROOT / DEFAULT_CELL_ANNO

from _spatial_pattern_utils import SHAPE_DISPLAY, SHAPE_FUNCS

OUT_ROOT = REPO_ROOT / DEFAULT_OUTPUT_DIR / "vary_shape"

grid = make_hex_grid()
barcodes_by_type = load_or_make_barcodes_by_type(CELL_ANNO, seed=SEED)
print("REPO_ROOT:", REPO_ROOT)
print("Grid spots:", len(grid))
print("Clone sizes:", TARGET_PER_TYPE)
print("Input cell_anno:", CELL_ANNO if CELL_ANNO.exists() else "(missing → synthetic barcodes)")
print("Output root:", OUT_ROOT)


In [ ]:
pattern_dirs = {}
for i, (name, func) in enumerate(SHAPE_FUNCS.items()):
    if name in ("mixed_unstructured", "gradient_infiltration"):
        labels = func(grid, seed=SEED + i)
    else:
        labels = func(grid)
    display = SHAPE_DISPLAY[name]
    barcodes = map_barcodes_to_labels(labels, barcodes_by_type, seed=SEED + 100 + i)
    outdir = write_pattern_dir(OUT_ROOT / display, grid, labels, barcodes)
    pattern_dirs[display] = (outdir, labels)
    print(f"{display:22s} -> {outdir}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (display, (_, labels)) in zip(axes.ravel(), pattern_dirs.items()):
    plot_spatial(ax, grid, labels, title=display.replace("_", " "))
axes[0, 0].legend(loc="upper left", fontsize=7, markerscale=2)
fig.suptitle("Varying clonal shapes", fontsize=12)
fig.tight_layout()
OUT_ROOT.mkdir(parents=True, exist_ok=True)
fig_path = OUT_ROOT / "vary_shape_overview.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Saved overview figure:", fig_path)
plt.show()
